# Construction Site Safety — Train YOLO trên Google Colab

Notebook này giúp:
1. Clone dự án và cài đặt thư viện
2. (Tùy chọn) Chạy Data Augmentation cho dataset
3. Train YOLOv26 với dataset PPE (7 class)
4. Tải file model `.pt` về máy

**Lưu ý:** Ứng dụng giám sát (giao diện desktop) vẫn chạy trên máy bạn; Colab chỉ dùng để train model.

## 1. Clone dự án và cài đặt

In [ ]:
# Cách A: Clone từ GitHub (thay YOUR_USERNAME và tên repo thực tế)
!git clone https://github.com/YOUR_USERNAME/Contruction-Site-Safety.git /content/Contruction
%cd /content/Contruction

In [ ]:
# Cách B: Upload ZIP dự án — chọn file ZIP rồi chạy ô này
from google.colab import files
import zipfile, os
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall('/content')
        print('Đã giải nén. Nội dung:', os.listdir('/content'))
        break
%cd /content/Contruction

In [ ]:
!pip install -q ultralytics opencv-python numpy Pillow
import sys
sys.path.insert(0, '/content/Contruction')

## 2. Chuẩn bị dataset (YOLO format)

Cấu trúc: `/content/dataset/train/images`, `train/labels`, `val/images`, `val/labels`. Upload ZIP dataset rồi giải nén vào `/content/dataset`.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        os.makedirs('/content/dataset', exist_ok=True)
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall('/content/dataset')
        print('Đã giải nén dataset.')
        break

## 3. (Tùy chọn) Data Augmentation

In [ ]:
import os
import sys
sys.path.insert(0, '/content/Contruction')
train_img = '/content/dataset/train/images'
train_lbl = '/content/dataset/train/labels'
out_aug = '/content/dataset/train_aug'
if os.path.isdir(train_img) and os.path.isdir(train_lbl):
    import subprocess
    subprocess.run([
        sys.executable, '-m', 'src.training.prepare_augmented_dataset',
        '--images', train_img, '--labels', train_lbl, '--out', out_aug,
        '--num', '2', '--copy-original'
    ], cwd='/content/Contruction', check=True)
else:
    print('Bỏ qua: chưa có train/images hoặc train/labels.')

## 4. Tạo data.yaml và Train YOLO (Runtime → Change runtime type → GPU)

In [ ]:
yaml_content = '''
path: /content/dataset
train: train/images
val: val/images
names:
  0: Gloves
  1: Helmet
  2: Non-Helmet
  3: Person
  4: Shoes
  5: Vest
  6: bare-arms
'''
with open('/content/Contruction/data_colab.yaml', 'w') as f:
    f.write(yaml_content.strip())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
model.train(data='/content/Contruction/data_colab.yaml', epochs=50, imgsz=640, batch=16, device=0, project='/content/Contruction/runs', name='ppe_detect', exist_ok=True)

## 5. Tải best.pt về máy

In [ ]:
from google.colab import files
import os
best = '/content/Contruction/runs/ppe_detect/weights/best.pt'
if os.path.isfile(best):
    files.download(best)
else:
    print('Chưa có best.pt.')